# Nettoyage des données Eurostat
Dette publique, comptes publics trimestriels et taux à 10 ans, UE 27 et agrégats, depuis 2000.

In [ ]:
import os
import pandas as pd

DOSSIER_BRUT = "donnees"
DOSSIER_PROPRE = "donnees_propres"
os.makedirs(DOSSIER_PROPRE, exist_ok=True)

URLS = {
    "dette": "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/gov_10q_ggdebt/Q.GD.S13.PC_GDP+MIO_EUR.?format=SDMX-CSV&startPeriod=2000-Q1",
    "comptes": "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/gov_10q_ggnfa/Q.PC_GDP.SCA.S13.B9+TR+TE+D41PAY.?format=SDMX-CSV&startPeriod=2000-Q1",
    "taux": "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/irt_lt_mcby_m/M.MCBY.?format=SDMX-CSV&startPeriod=2000-01",
}

## Chargement
Fichier local s'il existe dans `donnees/`, sinon téléchargement depuis l'API.

In [ ]:
def charger(nom):
    chemin = os.path.join(DOSSIER_BRUT, f"{nom}.csv")
    source = chemin if os.path.exists(chemin) else URLS[nom]
    df = pd.read_csv(source)
    print(f"{nom} : {df.shape[0]} lignes, source = {source}")
    return df

dette_brut = charger("dette")
comptes_brut = charger("comptes")
taux_brut = charger("taux")
dette_brut.head()

## Référentiel pays

In [ ]:
PAYS = {
    "AT": ("Autriche", True), "BE": ("Belgique", True), "BG": ("Bulgarie", True),
    "CY": ("Chypre", True), "CZ": ("Tchéquie", False), "DE": ("Allemagne", True),
    "DK": ("Danemark", False), "EE": ("Estonie", True), "EL": ("Grèce", True),
    "ES": ("Espagne", True), "FI": ("Finlande", True), "FR": ("France", True),
    "HR": ("Croatie", True), "HU": ("Hongrie", False), "IE": ("Irlande", True),
    "IT": ("Italie", True), "LT": ("Lituanie", True), "LU": ("Luxembourg", True),
    "LV": ("Lettonie", True), "MT": ("Malte", True), "NL": ("Pays-Bas", True),
    "PL": ("Pologne", False), "PT": ("Portugal", True), "RO": ("Roumanie", False),
    "SE": ("Suède", False), "SI": ("Slovénie", True), "SK": ("Slovaquie", True),
}
AGREGATS = {
    "EU27_2020": "UE 27",
    "EA20": "Zone euro",
    "EA21": "Zone euro",
    "EA": "Zone euro",
}

pays = pd.DataFrame(
    [(c, n, ze, "Pays") for c, (n, ze) in PAYS.items()]
    + [(c, n, True if c != "EU27_2020" else False, "Agrégat") for c, n in AGREGATS.items()],
    columns=["Code", "Pays", "Zone_euro", "Type"],
)
pays

## Fonctions de nettoyage

In [ ]:
def nettoyer(df, col_indicateur, noms, freq):
    df = df[["geo", "TIME_PERIOD", col_indicateur, "OBS_VALUE"]].copy()

    # valeurs numériques, les vides et les ':' deviennent NaN
    df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    # périmètre : UE 27 + agrégats
    df = df[df["geo"].isin(pays["Code"])]

    # un indicateur par colonne
    df = df.pivot_table(index=["geo", "TIME_PERIOD"], columns=col_indicateur,
                        values="OBS_VALUE", aggfunc="first").reset_index()
    df.columns.name = None
    df = df.rename(columns=noms)

    # date au premier jour de la période
    if freq == "Q":
        df["Date"] = pd.PeriodIndex(df["TIME_PERIOD"].str.replace("-Q", "Q"), freq="Q").to_timestamp()
    else:
        df["Date"] = pd.to_datetime(df["TIME_PERIOD"] + "-01")
    df["Annee"] = df["Date"].dt.year
    df["Trimestre"] = df["Annee"].astype(str) + "-T" + df["Date"].dt.quarter.astype(str)

    df = df.rename(columns={"geo": "Code"}).drop(columns="TIME_PERIOD")
    df = df.merge(pays[["Code", "Pays"]], on="Code", how="left")

    valeurs = [c for c in noms.values() if c in df.columns]
    df = df.dropna(subset=valeurs, how="all")
    return df.sort_values(["Code", "Date"]).reset_index(drop=True)

## Dette (stock, % du PIB et M€)

In [ ]:
dette = nettoyer(dette_brut, "unit",
                 {"PC_GDP": "Dette_pct_PIB", "MIO_EUR": "Dette_MEUR"}, "Q")
dette = dette[["Code", "Pays", "Date", "Annee", "Trimestre", "Dette_pct_PIB", "Dette_MEUR"]]
dette.tail()

## Comptes publics (flux, % du PIB, CVS-CJO)

In [ ]:
comptes = nettoyer(comptes_brut, "na_item",
                   {"B9": "Solde", "TR": "Recettes", "TE": "Depenses", "D41PAY": "Interets"}, "Q")
comptes["Solde_primaire"] = (comptes["Solde"] + comptes["Interets"]).round(2)
comptes = comptes[["Code", "Pays", "Date", "Annee", "Trimestre",
                   "Solde", "Recettes", "Depenses", "Interets", "Solde_primaire"]]
comptes.tail()

## Taux à 10 ans (mensuel) + moyenne trimestrielle

In [ ]:
taux_brut["int_rt"] = "Taux_10a"
taux = nettoyer(taux_brut, "int_rt", {"Taux_10a": "Taux_10a"}, "M")
taux = taux[["Code", "Pays", "Date", "Annee", "Trimestre", "Taux_10a"]]

taux_trim = (taux.groupby(["Code", "Pays", "Annee", "Trimestre"], as_index=False)["Taux_10a"].mean().round(3).round(3)
             .rename(columns={"Taux_10a": "Taux_10a_moy"}))
taux_trim["Date"] = pd.PeriodIndex(taux_trim["Trimestre"].str.replace("-T", "Q"), freq="Q").to_timestamp()
taux_trim = taux_trim[["Code", "Pays", "Date", "Annee", "Trimestre", "Taux_10a_moy"]]
taux.tail()

## Contrôles

In [ ]:
for nom, df in {"dette": dette, "comptes": comptes, "taux": taux}.items():
    print(f"--- {nom} : {len(df)} lignes, {df['Code'].nunique()} zones, "
          f"{df['Date'].min():%Y-%m} -> {df['Date'].max():%Y-%m}")
    print("valeurs manquantes :", df.isna().sum()[df.isna().sum() > 0].to_dict())
    print("zones absentes :", sorted(set(pays["Code"]) - set(df["Code"])))
    print("doublons :", df.duplicated(["Code", "Date"]).sum())

In [ ]:
# vérification rapide sur la France
print(dette[dette["Code"] == "FR"].tail(4)[["Trimestre", "Dette_pct_PIB"]])
print(comptes[comptes["Code"] == "FR"].tail(4)[["Trimestre", "Solde", "Interets"]])
print(taux[taux["Code"] == "FR"].tail(3)[["Date", "Taux_10a"]])

## Export Excel

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

TABLES = {
    "Dette": dette,
    "Comptes": comptes,
    "Taux": taux,
    "Taux_trimestriel": taux_trim,
    "Pays": pays,
}
FORMATS = {
    "Dette_pct_PIB": "0.0", "Dette_MEUR": "#,##0", "Solde": "0.0", "Recettes": "0.0",
    "Depenses": "0.0", "Interets": "0.0", "Solde_primaire": "0.0",
    "Taux_10a": "0.00", "Taux_10a_moy": "0.00", "Date": "dd/mm/yyyy",
}

def mettre_en_forme(ws, df):
    for i, col in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=i)
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="6E13EE")
        cell.alignment = Alignment(horizontal="center")
        ws.column_dimensions[get_column_letter(i)].width = max(12, len(col) + 4)
        if col in FORMATS:
            for row in ws.iter_rows(min_row=2, min_col=i, max_col=i):
                row[0].number_format = FORMATS[col]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

def exporter(nom, df, chemin):
    with pd.ExcelWriter(chemin, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=nom, index=False)
        mettre_en_forme(writer.sheets[nom], df)
    print(f"{chemin} : {len(df)} lignes")

# un fichier par table
for nom, df in TABLES.items():
    exporter(nom, df, os.path.join(DOSSIER_PROPRE, f"{nom.lower()}_propre.xlsx"))

# un classeur avec toutes les tables
chemin = os.path.join(DOSSIER_PROPRE, "donnees_propres.xlsx")
with pd.ExcelWriter(chemin, engine="openpyxl") as writer:
    for nom, df in TABLES.items():
        df.to_excel(writer, sheet_name=nom, index=False)
        mettre_en_forme(writer.sheets[nom], df)
print(f"{chemin} : {len(TABLES)} feuilles")